In [ ]:
"""
ml_training.py
=============================================================
Training model prediksi gaji IT Indonesia dari data JobStreet.
Jalankan di Google Colab atau laptop — BUKAN di GitHub Actions.

Tujuan: Membantu job seeker memprediksi estimasi gaji
        berdasarkan kategori, lokasi, skill, dan tipe kerja.

Output (commit ke folder models/ di GitHub):
  - models/model_gaji.pkl
  - models/model_metadata_gaji.pkl

Cara pakai di Colab:
  1. Upload file ini ke Colab
  2. pip install psycopg2-binary scikit-learn pandas joblib
  3. Set env: SUPABASE_HOST, SUPABASE_PASSWORD
  4. python ml_training.py
  5. Download folder models/ lalu commit ke GitHub
=============================================================
"""

import os
import joblib
import numpy as np
import pandas as pd
import psycopg2
from sklearn.ensemble        import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model    import Ridge
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing   import StandardScaler, OneHotEncoder
from sklearn.compose         import ColumnTransformer
from sklearn.pipeline        import Pipeline
from sklearn.metrics         import mean_absolute_error, r2_score
from google.colab import userdata


# ─── KONFIGURASI ────────────────────────────────────────────────

GAJI_MIN_VALID = 1_000_000    # Rp 1 juta  — batas bawah masuk akal
GAJI_MAX_VALID = 80_000_000   # Rp 80 juta — batas atas (hapus outlier ekstrem)


# ─── 1. AMBIL DATA ──────────────────────────────────────────────

def ambil_data() -> pd.DataFrame:
    print("[DATA] Mengambil data dari Supabase...")

    conn = psycopg2.connect(
        host     = userdata.get('SUPABASE_HOST'),
        port     = userdata.get('SUPABASE_PORT'),
        dbname   = userdata.get('SUPABASE_DB'),
        user     = userdata.get('SUPABASE_USER'),
        password = userdata.get('SUPABASE_PASSWORD')
    )

    query = """
        SELECT
            f.gaji_midpoint,
            f.jumlah_skill,
            f.posisi,
            CAST(f.is_active AS INTEGER)         AS is_active_int,

            k.kategori_besar,
            k.tipe_kerja_std,
            k.work_arrangement_std,

            l.provinsi_standar,
            CAST(l.is_remote AS INTEGER)         AS is_remote,

            p.ukuran_perusahaan,
            COALESCE(p.rating, 0)                AS company_rating,

            w.bulan,
            w.hari_minggu,

            f.skill_string

        FROM fact_lowongan f
        JOIN dim_klasifikasi k ON f.dim_klasifikasi_id = k.id
        JOIN dim_lokasi       l ON f.dim_lokasi_id      = l.id
        JOIN dim_perusahaan   p ON f.dim_perusahaan_id  = p.id
        JOIN dim_waktu        w ON f.dim_waktu_id        = w.id
        WHERE f.gaji_disclosed = true
          AND f.gaji_midpoint IS NOT NULL
          AND f.gaji_midpoint > 0;
    """

    df = pd.read_sql(query, conn)
    conn.close()
    print(f"  -> {len(df)} baris data diambil")
    return df


# ─── 2. FEATURE ENGINEERING ─────────────────────────────────────


def tambah_fitur_seniority(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ekstrak level seniority dari judul posisi.
    "Senior Data Engineer" → is_senior=1
    "Junior Frontend Developer" → is_junior=1
    Level seniority sangat berkorelasi dengan gaji.
    """
    posisi_lower = df["posisi"].fillna("").str.lower()

    df["is_senior"]  = posisi_lower.str.contains(
        r"senior|sr\.|sr |lead|principal|staff|expert|specialist",
        regex=True, case=False
    ).astype(int)

    df["is_junior"]  = posisi_lower.str.contains(
        r"junior|jr\.|jr |fresh|entry|associate|intern|magang",
        regex=True, case=False
    ).astype(int)

    df["is_manager"] = posisi_lower.str.contains(
        r"manager|head|director|vp |vice president|chief|cto|cio",
        regex=True, case=False
    ).astype(int)

    df["is_lead"]    = posisi_lower.str.contains(
        r"\blead\b|tech lead|team lead",
        regex=True, case=False
    ).astype(int)

    n_senior  = df["is_senior"].sum()
    n_junior  = df["is_junior"].sum()
    n_manager = df["is_manager"].sum()
    print(f"  [SENIORITY] Senior: {n_senior} | Junior: {n_junior} | Manager: {n_manager}")

    return df

def tambah_skill_diversity(df: pd.DataFrame) -> pd.DataFrame:
    """
    Hitung berapa banyak KATEGORI skill yang dimiliki loker.
    Mapping kategori didefinisikan langsung di sini agar tidak
    bergantung pada file transformer.py.
    """
    KATEGORI_SKILL = {
        "Python": "Programming", "SQL": "Programming", "Java": "Programming",
        "JavaScript": "Programming", "TypeScript": "Programming", "PHP": "Programming",
        "Kotlin": "Programming", "Swift": "Programming", "Go": "Programming",
        "R": "Programming", "Scala": "Programming", "C++": "Programming",
        "React": "Framework", "Vue": "Framework", "Angular": "Framework",
        "Django": "Framework", "FastAPI": "Framework", "Flask": "Framework",
        "Spring Boot": "Framework", "Laravel": "Framework", "Node.js": "Framework",
        "Next.js": "Framework",
        "Pandas": "Data & ML", "NumPy": "Data & ML", "TensorFlow": "Data & ML",
        "PyTorch": "Data & ML", "Scikit-learn": "Data & ML",
        "Machine Learning": "Data & ML", "Deep Learning": "Data & ML",
        "NLP": "Data & ML", "Computer Vision": "Data & ML",
        "Data Analysis": "Data & ML", "Power BI": "Data & ML",
        "Tableau": "Data & ML", "Looker": "Data & ML",
        "PostgreSQL": "Database", "MySQL": "Database", "MongoDB": "Database",
        "Redis": "Database", "Elasticsearch": "Database",
        "BigQuery": "Database", "Snowflake": "Database",
        "AWS": "Cloud & DevOps", "GCP": "Cloud & DevOps", "Azure": "Cloud & DevOps",
        "Docker": "Cloud & DevOps", "Kubernetes": "Cloud & DevOps",
        "Terraform": "Cloud & DevOps", "CI/CD": "Cloud & DevOps",
        "Git": "Cloud & DevOps", "GitHub": "Cloud & DevOps", "Linux": "Cloud & DevOps",
        "Apache Spark": "Data Engineering", "Apache Kafka": "Data Engineering",
        "Airflow": "Data Engineering", "dbt": "Data Engineering",
        "ETL": "Data Engineering", "Data Warehouse": "Data Engineering",
    }

    def hitung_diversity(skill_string):
        if not skill_string:
            return 0
        skills = [s.strip() for s in skill_string.split(",") if s.strip()]
        kategori_unik = set(
            KATEGORI_SKILL.get(s, "Lainnya") for s in skills
        )
        return len(kategori_unik)

    df["skill_diversity_score"] = df["skill_string"].apply(hitung_diversity)
    print(f"  [DIVERSITY] Rata-rata diversity score: {df['skill_diversity_score'].mean():.2f}")
    return df

def tambah_premium_skill(df: pd.DataFrame) -> pd.DataFrame:
    """
    Flag boolean: apakah loker mensyaratkan minimal 1 skill premium?
    Skill premium = skill yang dari feature importance sebelumnya
    terbukti berkorelasi kuat dengan gaji tinggi.
    """
    PREMIUM_SKILLS = [
        "Kubernetes", "Terraform", "Apache Spark",
        "Machine Learning", "Deep Learning", "GCP",
        "AWS", "Azure", "Airflow", "dbt",
        "Scala", "Go", "Rust",
    ]

    pattern = "|".join(PREMIUM_SKILLS)
    df["has_premium_skill"] = df["skill_string"].fillna("").str.contains(
        pattern, case=False, regex=True
    ).astype(int)

    n = df["has_premium_skill"].sum()
    pct = n / len(df) * 100
    print(f"  [PREMIUM] Loker dengan premium skill: {n} ({pct:.1f}%)")
    return df

def buat_fitur_skill(df: pd.DataFrame):
    """
    Ubah skill_string menjadi kolom boolean per skill populer.
    Skill yang muncul di >= 30 loker dijadikan fitur tersendiri.
    Prefix "fs_" (fitur_skill) dipakai agar tidak bentrok dengan kolom lain.
    """
    semua_skill = []
    for s in df["skill_string"].dropna():
        semua_skill.extend([x.strip() for x in s.split(",") if x.strip()])

    frek          = pd.Series(semua_skill).value_counts()
    skill_populer = frek[frek >= 30].index.tolist()
    print(f"  [SKILL] {len(skill_populer)} skill populer dijadikan fitur")

    import re
    for skill in skill_populer:
        # Prefix "fs_" agar tidak bentrok dengan kolom dataframe lain
        bersih = re.sub(r"[^a-z0-9]", "_", skill.lower()).strip("_")
        col    = "fs_" + bersih
        df[col] = df["skill_string"].fillna("").str.contains(
            skill, case=False, regex=False
        ).astype(int)

    return df, skill_populer


def preprocessing(df: pd.DataFrame):
    print("[PREP] Preprocessing data...")
    df = df.copy()

    # Hapus outlier gaji
    n_sebelum = len(df)
    df = df[
        (df["gaji_midpoint"] >= GAJI_MIN_VALID) &
        (df["gaji_midpoint"] <= GAJI_MAX_VALID)
    ].copy()
    print(f"  -> Outlier dihapus: {n_sebelum - len(df)} baris | Tersisa: {len(df)}")

    # Isi nilai kosong
    df["kategori_besar"]       = df["kategori_besar"].fillna("Lainnya")
    df["tipe_kerja_std"]       = df["tipe_kerja_std"].fillna("Tidak Disebutkan")
    df["work_arrangement_std"] = df["work_arrangement_std"].fillna("Tidak Disebutkan")
    df["provinsi_standar"]     = df["provinsi_standar"].fillna("Tidak Diketahui")
    df["ukuran_perusahaan"]    = df["ukuran_perusahaan"].fillna("Tidak Diketahui")
    df["jumlah_skill"]         = df["jumlah_skill"].fillna(0)
    df["is_remote"]            = df["is_remote"].fillna(False).astype(int)
    df["bulan"]                = df["bulan"].fillna(1)
    df["hari_minggu"]          = df["hari_minggu"].fillna(0)

    # Feature engineering: level seniority dari judul posisi
    df = tambah_fitur_seniority(df)

    df = tambah_skill_diversity(df)

    df = tambah_premium_skill(df)

    # Feature engineering dari skill
    df, skill_populer = buat_fitur_skill(df)

    # Definisi kolom fitur
    num_cols = ["jumlah_skill", "skill_diversity_score", "is_remote", "company_rating", "is_senior", "is_junior", "is_manager", "is_lead"]
    num_cols += [c for c in df.columns if c.startswith("fs_")]

    cat_cols = [
        "kategori_besar",
        "tipe_kerja_std",
        "work_arrangement_std",
        "provinsi_standar",
        "ukuran_perusahaan",
    ]

    X = df[num_cols + cat_cols]
    y = df["gaji_midpoint"].values

    # Preprocessor pipeline
    preprocessor = ColumnTransformer(transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ])

    print(f"  -> Fitur: {len(num_cols)} numerik + {len(cat_cols)} kategorikal")
    print(f"  -> Total sampel training: {len(X)}")

    return X, y, preprocessor, num_cols, cat_cols, skill_populer


# ─── 3. TRAINING ────────────────────────────────────────────────

def train(X, y, preprocessor):
    print("\n[TRAIN] Membandingkan model...")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    kandidat = {
        "Random Forest": RandomForestRegressor(
            n_estimators=300, max_depth=15,
            min_samples_leaf=3, max_features="sqrt",
            random_state=42
        ),
        "Gradient Boosting": GradientBoostingRegressor(
            n_estimators=300, max_depth=6,
            learning_rate=0.03, subsample=0.8,
            min_samples_leaf=3, random_state=42
        ),
        "Ridge Regression": Ridge(alpha=0.5),
    }

    hasil = {}
    for nama, model in kandidat.items():
        pipe = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model),
        ])
        cv = cross_val_score(
            pipe, X_train, y_train,
            cv=5, scoring="neg_mean_absolute_error"
        )
        mae_cv = -cv.mean()
        hasil[nama] = {"pipe": pipe, "mae_cv": mae_cv}
        print(f"  {nama:25s}: CV MAE = Rp {mae_cv:>12,.0f}")

    # Pilih model terbaik (MAE terkecil)
    best_name = min(hasil, key=lambda k: hasil[k]["mae_cv"])
    best_pipe = hasil[best_name]["pipe"]
    print(f"\n  -> Model terbaik: {best_name}")

    # Training final dengan semua data train
    best_pipe.fit(X_train, y_train)

    # Evaluasi di test set
    y_pred = best_pipe.predict(X_test)
    mae    = mean_absolute_error(y_test, y_pred)
    r2     = r2_score(y_test, y_pred)

    print(f"\n[EVAL] Evaluasi di Test Set (data yang tidak pernah dilihat model):")
    print(f"  MAE : Rp {mae:,.0f}")
    print(f"        Artinya rata-rata selisih prediksi vs gaji aktual sebesar Rp {mae:,.0f}")
    print(f"  R²  : {r2:.3f}")
    print(f"        Artinya model menjelaskan {r2*100:.1f}% variansi gaji")

    print(f"\n[CONTOH] 5 prediksi vs aktual:")
    for i in range(min(5, len(y_test))):
        selisih = abs(y_test[i] - y_pred[i])
        print(f"  Aktual: Rp {y_test[i]:>12,.0f} | "
              f"Prediksi: Rp {y_pred[i]:>12,.0f} | "
              f"Selisih: Rp {selisih:>10,.0f}")

    return best_pipe, best_name, mae, r2


# ─── 4. FEATURE IMPORTANCE ──────────────────────────────────────

def feature_importance(pipe, num_cols, cat_cols):
    """
    Tampilkan fitur paling berpengaruh.
    Berguna untuk bab analisis TA — faktor apa yang paling
    mempengaruhi gaji IT di Indonesia.
    """
    try:
        model = pipe.named_steps["model"]
        if not hasattr(model, "feature_importances_"):
            print("\n[INFO] Ridge Regression tidak support feature importance")
            return

        ohe      = pipe.named_steps["preprocessor"].named_transformers_["cat"]
        ohe_cols = list(ohe.get_feature_names_out(cat_cols))
        all_cols = num_cols + ohe_cols

        imp = pd.Series(model.feature_importances_, index=all_cols)
        imp = imp.sort_values(ascending=False)

        print("\n[IMPORTANCE] Top 15 faktor yang paling mempengaruhi gaji:")
        print("-" * 55)
        for fitur, nilai in imp.head(15).items():
            bar = "█" * int(nilai * 300)
            print(f"  {fitur[:38]:<38} {nilai:.4f} {bar}")

    except Exception as e:
        print(f"[INFO] Feature importance gagal: {e}")


# ─── 5. SIMPAN ──────────────────────────────────────────────────

def simpan(pipe, metadata: dict):
    os.makedirs("models", exist_ok=True)
    joblib.dump(pipe,     "models/model_gaji.pkl")
    joblib.dump(metadata, "models/model_metadata_gaji.pkl")

    print(f"\n[SAVE] File tersimpan:")
    print(f"  models/model_gaji.pkl")
    print(f"  models/model_metadata_gaji.pkl")
    print(f"\n  Langkah selanjutnya:")
    print(f"  1. Download folder models/")
    print(f"  2. git add models/ && git commit -m 'add salary prediction model'")
    print(f"  3. git push → inference otomatis berjalan di pipeline")


# ─── MAIN ────────────────────────────────────────────────────────

def main():
    print("=" * 55)
    print("  ML TRAINING — Prediksi Gaji IT Indonesia")
    print("=" * 55)

    df                                              = ambil_data()
    X, y, preprocessor, num_cols, cat_cols, skills = preprocessing(df)
    best_pipe, best_name, mae, r2                   = train(X, y, preprocessor)

    feature_importance(best_pipe, num_cols, cat_cols)

    metadata = {
        "model_name"    : best_name,
        "n_samples"     : len(X),
        "num_cols"      : num_cols,
        "cat_cols"      : cat_cols,
        "skill_populer" : skills,
        "mae"           : mae,
        "r2"            : r2,
        "gaji_min_valid": GAJI_MIN_VALID,
        "gaji_max_valid": GAJI_MAX_VALID,
        "version"       : "1.0",
    }
    simpan(best_pipe, metadata)

    print("\n" + "=" * 55)
    print(f"  Training selesai!")
    print(f"  Model : {best_name}")
    print(f"  MAE   : Rp {mae:,.0f}")
    print(f"  R²    : {r2:.3f}")
    print("=" * 55)


if __name__ == "__main__":
    main()

  ML TRAINING — Prediksi Gaji IT Indonesia
[DATA] Mengambil data dari Supabase...


/tmp/ipykernel_1986/2699934472.py:89: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


  -> 965 baris data diambil
[PREP] Preprocessing data...
  -> Outlier dihapus: 5 baris | Tersisa: 960
  [SENIORITY] Senior: 143 | Junior: 46 | Manager: 86
  [DIVERSITY] Rata-rata diversity score: 1.55
  [PREMIUM] Loker dengan premium skill: 231 (24.1%)
  [SKILL] 27 skill populer dijadikan fitur
  -> Fitur: 35 numerik + 5 kategorikal
  -> Total sampel training: 960

[TRAIN] Membandingkan model...
  Random Forest            : CV MAE = Rp    3,268,880
  Gradient Boosting        : CV MAE = Rp    3,309,204
  Ridge Regression         : CV MAE = Rp    3,550,648

  -> Model terbaik: Random Forest

[EVAL] Evaluasi di Test Set (data yang tidak pernah dilihat model):
  MAE : Rp 3,777,446
        Artinya rata-rata selisih prediksi vs gaji aktual sebesar Rp 3,777,446
  R²  : 0.272
        Artinya model menjelaskan 27.2% variansi gaji

[CONTOH] 5 prediksi vs aktual:
  Aktual: Rp    8,900,000 | Prediksi: Rp    8,777,223 | Selisih: Rp    122,777
  Aktual: Rp    6,600,000 | Prediksi: Rp    7,193,818 | 